# Avro Basics — 10: Avro Compression


Avro supports multiple compression codecs. The right choice depends on
your read/write frequency, CPU budget, and storage costs.

Topics: uncompressed/snappy/deflate/bzip2/xz benchmark, Avro-specific settings,
compression for Kafka vs storage, measuring actual compression ratio per data type.


In [1]:
import os, time, pathlib, shutil, random, datetime, json as pyjson
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *

MASTER   = 'spark://spark-master:7077'
DATA_DIR = '/workspace/data/avro_basics'
pathlib.Path(DATA_DIR).mkdir(parents=True, exist_ok=True)

spark = (
    SparkSession.builder
    .appName('avro-basics')
    .master(MASTER)
    .config('spark.executor.memory', '2g')
    .config('spark.driver.memory',   '1g')
    .config('spark.executor.cores',  '2')
    .config('spark.shuffle.sort.bypassMergeThreshold', '0')
    .getOrCreate()
)
spark.sparkContext.setLogLevel('WARN')
print(f"Spark {spark.version} | Avro JAR: spark-avro_2.13-4.0.2")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/10 13:08:28 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark 4.0.2 | Avro JAR: spark-avro_2.13-4.0.2


## Step 1 — Generate Benchmark Dataset



In [2]:

import random, pathlib
random.seed(42)
N = 300_000

CATEGORIES = ["Electronics","Clothing","Books","Food","Sports","Furniture","Toys"]
COUNTRIES  = ["US","UK","DE","FR","JP","CA","AU","BR","IN","MX"]

df_bench = spark.range(N).select(
    F.col("id").alias("event_id"),
    (F.col("id") % 50000 + 1).alias("user_id"),
    F.element_at(F.array([F.lit(c) for c in CATEGORIES]),(F.col("id")%7+1).cast("int")).alias("category"),
    F.element_at(F.array([F.lit(c) for c in COUNTRIES]), (F.col("id")%10+1).cast("int")).alias("country"),
    F.round(F.rand(42)*500+10, 2).alias("revenue"),
    F.round(F.rand(43)*9+1, 2).alias("score"),
    F.concat(F.lit("SKU-"), (F.col("id")%10000).cast("string")).alias("product_sku"),
    F.when(F.col("id")%3==0, F.lit(True)).otherwise(F.lit(False)).alias("is_purchase"),
)
print(f"Benchmark dataset: {N:,} rows, {len(df_bench.columns)} columns")
df_bench.show(3)


Benchmark dataset: 300,000 rows, 8 columns


[Stage 0:>                                                          (0 + 1) / 1]

+--------+-------+-----------+-------+-------+-----+-----------+-----------+
|event_id|user_id|   category|country|revenue|score|product_sku|is_purchase|
+--------+-------+-----------+-------+-------+-----+-----------+-----------+
|       0|      1|Electronics|     US| 319.59| 8.22|      SKU-0|       true|
|       1|      2|   Clothing|     UK|  264.8| 6.91|      SKU-1|      false|
|       2|      3|      Books|     DE| 426.26| 3.26|      SKU-2|      false|
+--------+-------+-----------+-------+-------+-----+-----------+-----------+
only showing top 3 rows


## Step 2 — Full Codec Benchmark



In [3]:

import glob as glib

codecs = ["uncompressed", "snappy", "deflate", "bzip2"]

# Try xz if available
try:
    test_path = f"{DATA_DIR}/test_xz"
    df_bench.limit(100).write.format("avro").option("compression","xz").mode("overwrite").save(test_path)
    codecs.append("xz")
except:
    pass

results = {}
for codec in codecs:
    path = f"{DATA_DIR}/avro_{codec}"
    # Write
    t0 = time.time()
    df_bench.write.format("avro").mode("overwrite").option("compression",codec).save(path)
    t_write = time.time() - t0
    # Size
    files   = glib.glob(f"{path}/*.avro")
    size_mb = sum(pathlib.Path(f).stat().st_size for f in files) / 1024/1024
    # Read
    tr = []
    for _ in range(3):
        t0=time.time(); spark.read.format("avro").load(path).agg(F.sum("revenue")).collect(); tr.append(time.time()-t0)
    results[codec] = {"write":t_write, "size":size_mb, "read":sorted(tr)[1]}

base = results["uncompressed"]
print(f"{'Codec':<15} {'Write':>8} {'Size MB':>10} {'vs uncomp':>10} {'Read':>8}")
print("-" * 58)
for codec, r in results.items():
    ratio = f"{r['size']/base['size']:.2f}x"
    print(f"  {codec:<13} {r['write']:>6.2f}s {r['size']:>8.1f} MB {ratio:>10} {r['read']:>6.3f}s")


Codec              Write    Size MB  vs uncomp     Read
----------------------------------------------------------
  uncompressed    1.24s     13.9 MB      1.00x  0.464s
  snappy          0.78s      6.9 MB      0.50x  0.337s
  deflate         0.84s      4.8 MB      0.35x  0.323s
  bzip2           1.47s      3.4 MB      0.24x  0.462s
  xz              3.29s      3.7 MB      0.26x  0.482s


## Step 3 — Data Type Compression Efficiency



In [4]:

# Different data types compress very differently in Avro
low_card  = df_bench.select("event_id","category","country","is_purchase")    # strings + bool
high_card = df_bench.select("event_id","user_id","revenue","score","product_sku")  # numeric + high card string

for label, sub_df in [("Low cardinality (category/country)", low_card),
                       ("High cardinality (revenue/sku)",     high_card)]:
    for codec in ["uncompressed","snappy","deflate"]:
        path = f"{DATA_DIR}/dtype_{codec}_{label[:3].lower()}"
        sub_df.write.format("avro").option("compression",codec).mode("overwrite").save(path)
        files = glib.glob(f"{path}/*.avro")
        mb = sum(pathlib.Path(f).stat().st_size for f in files)/1024/1024
        print(f"  {label[:35]:<35} {codec:<14} {mb:.1f} MB")


  Low cardinality (category/country)  uncompressed   4.8 MB
  Low cardinality (category/country)  snappy         1.9 MB
  Low cardinality (category/country)  deflate        1.2 MB
  High cardinality (revenue/sku)      uncompressed   9.9 MB
  High cardinality (revenue/sku)      snappy         5.4 MB
  High cardinality (revenue/sku)      deflate        3.7 MB


## Step 4 — Kafka vs Storage Recommendations



In [5]:

print("""
Compression recommendations by use case:

KAFKA MESSAGES (transport):
  snappy    ← recommended
  - Low CPU overhead (fast producer/consumer)
  - Good compression ratio for typical event data
  - Supported by all Kafka brokers natively

AVRO FILES (landing zone storage):
  snappy    ← for hot/frequently-read data
  deflate   ← for warm/periodically-read data
  bzip2     ← for cold/archival data (best ratio)

STREAMING MICRO-BATCHES:
  snappy    ← always (low latency, low CPU)

BATCH ANALYTICS STORAGE → Use Parquet instead!
  Parquet+zstd is 3-5x smaller than Avro+deflate
  and much faster for analytical queries.

deflate level (default 6, range 0-9):
  .option("deflate.level", "6")
  Higher = better compression, slower
""")



Compression recommendations by use case:

KAFKA MESSAGES (transport):
  snappy    ← recommended
  - Low CPU overhead (fast producer/consumer)
  - Good compression ratio for typical event data
  - Supported by all Kafka brokers natively

AVRO FILES (landing zone storage):
  snappy    ← for hot/frequently-read data
  deflate   ← for warm/periodically-read data
  bzip2     ← for cold/archival data (best ratio)

STREAMING MICRO-BATCHES:
  snappy    ← always (low latency, low CPU)

BATCH ANALYTICS STORAGE → Use Parquet instead!
  Parquet+zstd is 3-5x smaller than Avro+deflate
  and much faster for analytical queries.

deflate level (default 6, range 0-9):
  .option("deflate.level", "6")
  Higher = better compression, slower



## Summary



In [6]:

print("""
Avro compression cheat sheet:

  codec          ratio    cpu    best for
  ───────────────────────────────────────
  uncompressed   1.0x     zero   debugging
  snappy         ~0.6x    low    Kafka, hot storage, streaming
  deflate        ~0.4x    medium cold storage, batch files
  bzip2          ~0.35x   high   archival
  xz             ~0.3x    high   maximum compression

Configuration:
  .option("compression", "snappy")
  .option("compression", "deflate")
  .option("deflate.level", "6")      # 0-9, default 6

  Or in spark-defaults.conf:
  spark.sql.avro.compression.codec = snappy
""")



Avro compression cheat sheet:

  codec          ratio    cpu    best for
  ───────────────────────────────────────
  uncompressed   1.0x     zero   debugging
  snappy         ~0.6x    low    Kafka, hot storage, streaming
  deflate        ~0.4x    medium cold storage, batch files
  bzip2          ~0.35x   high   archival
  xz             ~0.3x    high   maximum compression

Configuration:
  .option("compression", "snappy")
  .option("compression", "deflate")
  .option("deflate.level", "6")      # 0-9, default 6

  Or in spark-defaults.conf:
  spark.sql.avro.compression.codec = snappy

